In [ ]:
# 导入类型定义和LangGraph的START、END特殊节点
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
# 定义代理状态
class AgentState(TypedDict):
    number1: int       # 第一个数
    operation: str     # 运算符（"+" 或 "-"）
    number2: int       # 第二个数
    finalNumber: int   # 计算结果

In [ ]:
# 加法节点：将两个数相加
def adder(state:AgentState) -> AgentState:
    """This node adds the 2 numbers"""
    state["finalNumber"] = state["number1"] + state["number2"]

    return state

# 减法节点：将两个数相减
def subtractor(state:AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

# 路由函数：根据运算符决定下一个节点
def decide_next_node(state:AgentState) -> AgentState:
    """This node will select the next node of the graph"""

    if state["operation"] == "+":
        return "addition_operation"     # 走加法分支
    
    elif state["operation"] == "-":
        return "subtraction_operation"  # 走减法分支


In [ ]:
# 构建条件分支图
graph = StateGraph(AgentState)

# 添加加法和减法节点
graph.add_node("add_node", adder)
graph.add_node("subtract_node", subtractor)
graph.add_node("router", lambda state:state) # passthrough function
# 路由器节点（透传函数，不修改状态）

# 从起始节点进入路由器
graph.add_edge(START, "router") 

# 添加条件边：根据decide_next_node的返回值选择分支
graph.add_conditional_edges(
    "router",
    decide_next_node, 

    {
        # Edge: Node
        "addition_operation": "add_node",       # 加法分支
        "subtraction_operation": "subtract_node" # 减法分支
    }

)

# 两个分支都指向结束节点
graph.add_edge("add_node", END)
graph.add_edge("subtract_node", END)

# 编译图
app = graph.compile()

In [ ]:
# 可视化图结构
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 使用TypedDict方式创建初始状态并调用图（10 - 5 = 5）
initial_state_1 = AgentState(number1 = 10, operation="-", number2 = 5)
print(app.invoke(initial_state_1))

In [ ]:
# This way still works!
# 也可以直接使用字典方式传入初始状态
result = app.invoke({"number1": 10, "operation": "-", "number2": 5})
print(result)